# Surrogate Evaluator Validity & β Penalty Calibration Study

**Research Questions**

1. *Does the `StaticSurrogateEvaluator` produce rankings that are meaningfully correlated with full microscopic simulation fitness scores?* If not, what structural modifications improve its predictive validity?
2. *What value of the underserved-passenger penalty coefficient β yields the most discriminating and stable fitness landscape for the metaheuristic search?*

**Motivation**

The surrogate evaluator is orders of magnitude faster than a full agent-based simulation and is the primary fitness signal during the GA search phase. Its value depends entirely on whether it *ranks* route configurations in the same order the microscopic simulator would. A scatter plot of existing results (Pearson r=0.21, Spearman ρ=0.16) suggests the current surrogate has very low rank-agreement — it explains roughly 4% of microscopic fitness variance. This study characterises why, and proposes targeted fixes.

Separately, the β penalty in `_calculate_results()` controls how harshly unserved passengers are penalised. Too soft a β makes the landscape flat; too aggressive a β dominates the signal and drowns out route quality differences.

---

## 0. Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr, pearsonr, kendalltau
from scipy.optimize import minimize_scalar
from sklearn.preprocessing import MinMaxScaler
from pathlib import Path
import warnings
import yaml
import pickle

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})

# ── Project imports ──────────────────────────────────────────────────────────
from utils.simulation import Simulation, SimulationResult, SimulationEvaluator, StaticSurrogateEvaluator
from utils.city_graph import CityGraph
from utils.direct_demand_sampler import DirectDemandSampler
from utils.route import Route
from utils_simplified import reuse_citygraph, reuse_ddm

# ── Paths — adjust to your run directory ─────────────────────────────────────
CONFIG_YAML   = 'configs/profile_p1.yaml'
CG_PKL        = 'cache/city_graph.pkl'
DDM_PKL       = 'cache/ddm.pkl'
RESULTS_DIR   = Path('results/calibration')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

with open(CONFIG_YAML) as f:
    BASE_CONFIG = yaml.safe_load(f)

print('Environment ready.')

In [ ]:
# Load pre-built city graph and demand model (expensive — reuse pickles)
cg  = reuse_citygraph(CG_PKL)
ddm = reuse_ddm(DDM_PKL)
print(f'CityGraph nodes : {cg.graph.number_of_nodes():,}')
print(f'CityGraph edges : {cg.graph.number_of_edges():,}')

---
## 1. Paired Evaluation Dataset

We need a set of route configurations evaluated by *both* evaluators under identical conditions. If you already have logged `SimulationResult` report files from a prior GA run you can load those directly (Section 1A). Otherwise Section 1B generates a fresh paired dataset.

### 1A  Load existing results from report files  *(skip if generating fresh)*

In [ ]:
# ── Adjust glob pattern to wherever your report files live ───────────────────
REPORT_DIR = Path('results/reports')

fitness_records  = []
surrogate_records = []

for fp in sorted(REPORT_DIR.glob('report_sim_*.txt')):
    try:
        r = SimulationResult.from_file(str(fp))
        if r.score_kind == 'fitness'   and r.fitness_score  is not None:
            fitness_records.append(r)
        elif r.score_kind == 'surrogate' and r.surrogate_cost is not None:
            surrogate_records.append(r)
    except Exception as e:
        print(f'  [skip] {fp.name}: {e}')

print(f'Loaded {len(fitness_records)} fitness results and {len(surrogate_records)} surrogate results.')

### 1B  Generate a fresh paired dataset  *(run if 1A is empty)*

We sample a diverse population of route configurations and evaluate every one with both evaluators, keeping the route objects constant. The `StaticSurrogateEvaluator` is constructed once (OD pairs are fixed); `SimulationEvaluator` runs full agent simulations.

In [ ]:
from utils.route import RouteGenerator

N_CONFIGS      = 40   # number of distinct route configurations
N_ROUTES       = 4    # routes per configuration
SURROGATE_OD   = 500  # OD samples used by the surrogate

route_gen = RouteGenerator(city_graph=cg, sampler=ddm, verbose=False)

def sample_route_system(n_routes: int = N_ROUTES) -> list:
    return [route_gen.generate(n_points=4) for _ in range(n_routes)]

route_systems = [sample_route_system() for _ in range(N_CONFIGS)]
print(f'Sampled {N_CONFIGS} route configurations.')

In [ ]:
from tqdm.notebook import tqdm

# Build evaluators — surrogate is constructed once so OD pairs are shared
sim_evaluator = SimulationEvaluator(
    config=BASE_CONFIG,
    city_graph=cg,
    travel_graph=None,
    demand_sampler=ddm
)

surrogate_evaluator = StaticSurrogateEvaluator(
    config=BASE_CONFIG,
    city_graph=cg,
    demand_sampler=ddm,
    num_samples=SURROGATE_OD
)

paired_records = []

for i, routes in enumerate(tqdm(route_systems, desc='Evaluating configurations')):
    surr = surrogate_evaluator.evaluate(routes)
    full = sim_evaluator.evaluate(routes)
    paired_records.append({
        'config_id'              : i,
        'surrogate_cost'         : surr.surrogate_cost,
        'surrogate_completed'    : surr.metrics.get('completed_routes', 0),
        'surrogate_samples'      : surr.metrics.get('surrogate_samples', SURROGATE_OD),
        'fleet_operational_cost' : surr.metrics.get('fleet_operational_cost', np.nan),
        'fitness_score'          : full.fitness_score,
        'completed_count'        : full.metrics.get('completed_count', 0),
        'incomplete_count'       : full.metrics.get('incomplete_count', 0),
        'mean_commute_time'      : full.metrics.get('mean_commute_time', np.nan),
        'std_commute_time'       : full.metrics.get('std_commute_time', np.nan),
        'sum_penalty_time'       : full.metrics.get('sum_penalty_time', np.nan),
        'equity_penalty'         : full.metrics.get('equity_penalty', np.nan),
    })

df = pd.DataFrame(paired_records)
df.to_csv(RESULTS_DIR / 'paired_evaluations.csv', index=False)
print(df.shape, '— saved.')
df.head()

---
## 2. Baseline Surrogate Validity

Before proposing changes we establish the baseline agreement between surrogate cost and microscopic fitness — replicating the scatter in the motivating figure.

In [ ]:
def correlation_report(x: np.ndarray, y: np.ndarray, label_x='Surrogate Cost', label_y='Fitness Score') -> dict:
    """Returns Pearson r, Spearman ρ, Kendall τ and rank-RMSE for two arrays."""
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    r,  p_r  = pearsonr(x, y)
    rho, p_s = spearmanr(x, y)
    tau, p_k = kendalltau(x, y)
    rank_x = stats.rankdata(x)
    rank_y = stats.rankdata(y)
    rank_rmse = np.sqrt(np.mean((rank_x - rank_y)**2))
    return dict(pearson_r=r, pearson_p=p_r,
                spearman_rho=rho, spearman_p=p_s,
                kendall_tau=tau, kendall_p=p_k,
                rank_rmse=rank_rmse, n=len(x))

baseline = correlation_report(
    df['surrogate_cost'].values,
    df['fitness_score'].values
)
print('=== Baseline Surrogate Validity ===')
for k, v in baseline.items():
    print(f'  {k:<20s}: {v:.4f}' if isinstance(v, float) else f'  {k:<20s}: {v}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: raw scatter
ax = axes[0]
ax.scatter(df['surrogate_cost'], df['fitness_score'], alpha=0.7, edgecolors='white', linewidth=0.4, s=60)
m, b = np.polyfit(df['surrogate_cost'].dropna(), df['fitness_score'].dropna(), 1)
xs = np.linspace(df['surrogate_cost'].min(), df['surrogate_cost'].max(), 100)
ax.plot(xs, m*xs + b, 'k--', lw=1.5, label=f"OLS (r={baseline['pearson_r']:.2f})")
ax.set_xlabel('Surrogate Cost (Static)')
ax.set_ylabel('Fitness Score (Microscopic)')
ax.set_title('Baseline: Raw Score Agreement')
ax.legend()

# Right: rank-rank scatter (more diagnostic)
ax = axes[1]
rank_surr = stats.rankdata(df['surrogate_cost'])
rank_fit  = stats.rankdata(df['fitness_score'])
ax.scatter(rank_surr, rank_fit, alpha=0.7, edgecolors='white', linewidth=0.4, s=60, color='steelblue')
ax.plot([1, len(df)], [1, len(df)], 'k--', lw=1.2, label='Perfect rank agreement')
ax.set_xlabel('Surrogate Rank (1=best route system)')
ax.set_ylabel('Fitness Rank (1=best route system)')
ax.set_title(f"Rank Agreement (ρ={baseline['spearman_rho']:.2f}, τ={baseline['kendall_tau']:.2f})")
ax.legend()

plt.suptitle('Surrogate vs Microscopic — Baseline Validity', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig1_baseline_validity.png', bbox_inches='tight')
plt.show()

---
## 3. Diagnosing the Surrogate Gap

We decompose the surrogate cost into its components and check which factors the microscopic simulator cares about that the surrogate currently ignores.

In [ ]:
# Derived surrogate features not currently in the score
df['surr_coverage_ratio'] = df['surrogate_completed'] / df['surrogate_samples'].replace(0, np.nan)
df['surr_cost_per_routed'] = df['surrogate_cost'] / df['surrogate_completed'].replace(0, np.nan)
df['surr_unreachable_count'] = df['surrogate_samples'] - df['surrogate_completed']

candidate_predictors = [
    'surrogate_cost',
    'surr_coverage_ratio',
    'surr_cost_per_routed',
    'fleet_operational_cost',
    'surr_unreachable_count',
]

target_components = [
    'fitness_score',
    'sum_penalty_time',
    'mean_commute_time',
    'std_commute_time',
    'completed_count',
    'incomplete_count',
]

corr_matrix = pd.DataFrame(index=candidate_predictors, columns=target_components, dtype=float)
for pred in candidate_predictors:
    for tgt in target_components:
        valid = df[[pred, tgt]].dropna()
        if len(valid) > 3:
            rho, _ = spearmanr(valid[pred], valid[tgt])
            corr_matrix.loc[pred, tgt] = rho

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.heatmap(corr_matrix.astype(float), annot=True, fmt='.2f', center=0,
            cmap='RdBu_r', vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title('Spearman ρ: Surrogate Features vs Microscopic Components', fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig2_diagnostic_heatmap.png', bbox_inches='tight')
plt.show()
print('\nKey finding: surr_coverage_ratio and surr_cost_per_routed may correlate more strongly')
print('with penalty_time and incomplete_count than raw surrogate_cost does.')

---
## 4. Improved Surrogate Score Formulations

We construct candidate composite scores from existing surrogate outputs, without re-running anything, and test which formulation best predicts microscopic fitness.

The general form tested is:

$$S_{\text{composite}}(\lambda, \gamma) = \frac{\text{total\_weight}}{\text{completed}^{\lambda}} + \gamma \cdot (\text{n\_samples} - \text{completed}) \cdot C_{\text{unreachable}}$$

where $\lambda$ controls the normalisation by coverage and $\gamma$ scales the explicit coverage penalty. The current evaluator implicitly uses $\lambda=0$, $\gamma=1$ with a hard-coded $C_{\text{unreachable}}=100{,}000$.

In [ ]:
from scipy.optimize import differential_evolution

sc = df['surrogate_cost'].values
completed = df['surrogate_completed'].values.clip(1)  # avoid /0
n_samples = df['surrogate_samples'].values
fleet_cost = df['fleet_operational_cost'].values
fitness = df['fitness_score'].values

def composite_score(params) -> np.ndarray:
    lambda_, gamma, delta = params
    routing_term  = sc / (completed ** lambda_)
    coverage_term = gamma * (n_samples - completed)
    fleet_term    = delta * fleet_cost
    return routing_term + coverage_term + fleet_term

def neg_spearman(params) -> float:
    """Objective: maximise Spearman ρ between composite score and fitness."""
    try:
        s = composite_score(params)
        if not np.all(np.isfinite(s)):
            return 0.0
        rho, _ = spearmanr(s, fitness)
        return -rho  # minimise negative ρ
    except Exception:
        return 0.0

# Search over λ ∈ [0, 2], γ ∈ [0, 5000], δ ∈ [0, 1]
bounds = [(0.0, 2.0), (0.0, 5000.0), (0.0, 1.0)]
result = differential_evolution(
    neg_spearman,
    bounds=bounds,
    seed=42,
    maxiter=300,
    popsize=18,
    tol=1e-6,
    polish=True,
    workers=1
)

best_lambda, best_gamma, best_delta = result.x
best_rho = -result.fun

print(f'Optimised composite surrogate:')
print(f'  λ (coverage normalisation exponent) = {best_lambda:.4f}')
print(f'  γ (unreachable OD penalty weight)    = {best_gamma:.2f}')
print(f'  δ (fleet operational cost weight)    = {best_delta:.6f}')
print(f'  Spearman ρ vs fitness                = {best_rho:.4f}  (baseline: {baseline["spearman_rho"]:.4f})')

In [ ]:
# Build comparison DataFrame of candidate formulations
formulations = {
    'Baseline (current)' : sc,
    'Coverage-normalised (λ=1)' : sc / completed,
    'Coverage-penalty (γ=mean_edge)': sc + df['surr_unreachable_count'].values * sc.mean() / n_samples,
    'Cost-per-routed + fleet' : (sc / completed) + fleet_cost * 0.01,
    'Optimised composite'    : composite_score([best_lambda, best_gamma, best_delta]),
}

comparison_rows = []
for name, scores in formulations.items():
    r  = correlation_report(scores, fitness, label_x=name)
    comparison_rows.append({'Formulation': name, **r})

df_comp = pd.DataFrame(comparison_rows)[['Formulation','pearson_r','spearman_rho','kendall_tau','rank_rmse']]
df_comp = df_comp.sort_values('spearman_rho', ascending=False).reset_index(drop=True)
print(df_comp.to_string(index=False, float_format='{:.4f}'.format))

In [ ]:
fig, axes = plt.subplots(1, len(formulations), figsize=(4.5 * len(formulations), 4.5))
if len(formulations) == 1:
    axes = [axes]

rank_fit = stats.rankdata(fitness)
for ax, (name, scores) in zip(axes, formulations.items()):
    rank_s = stats.rankdata(scores)
    rho, _ = spearmanr(scores, fitness)
    ax.scatter(rank_s, rank_fit, s=40, alpha=0.7, edgecolors='white', lw=0.3)
    ax.plot([1, len(df)], [1, len(df)], 'r--', lw=1, alpha=0.6)
    ax.set_title(f'{name}\nρ={rho:.3f}', fontsize=9)
    ax.set_xlabel('Surrogate Rank')
    ax.set_ylabel('Fitness Rank')

plt.suptitle('Rank Agreement: Baseline vs Improved Surrogate Formulations', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig3_formulation_comparison.png', bbox_inches='tight')
plt.show()

---
## 5. β Penalty Calibration — Bayesian Optimisation

Rather than a uniform grid sweep, we use Bayesian optimisation with a Gaussian Process surrogate (on a 1-D domain over β) to find the value that maximises **fitness landscape discriminability** — defined as the variance of fitness scores across a fixed set of route configurations.

A flat landscape (low variance) means the optimizer receives weak gradient signal; an over-peaked landscape (dominated by a single β-scaled term) loses signal about routing quality. We want β that maximises the signal-to-noise ratio of the fitness landscape, operationalised as:

$$\text{LDS}(\beta) = \frac{\sigma^2_{\text{fitness}}}{\overline{\text{sum\_penalty\_time}}}$$

i.e., fitness variance normalised by the mean penalty component, so β doesn't trivially maximise variance by inflating penalties.

In [ ]:
# ── Gaussian-Process Bayesian Optimisation over β ─────────────────────────────
# Pure-Python implementation using scipy — no external BO library required.

from scipy.spatial.distance import cdist

class GaussianProcessBO:
    """
    Minimal 1-D Gaussian Process Bayesian Optimiser.
    Kernel: Matérn 5/2 with automatic length-scale and noise variance.
    Acquisition: Expected Improvement (EI).
    """
    def __init__(self, bounds: tuple, n_init: int = 5, noise: float = 1e-4):
        self.bounds  = bounds
        self.n_init  = n_init
        self.noise   = noise
        self.X_obs   = np.empty((0, 1))
        self.y_obs   = np.empty(0)
        self.ls      = (bounds[1] - bounds[0]) / 4.0  # initial length scale
        self.var_f   = 1.0

    def _matern52(self, X1, X2):
        r = cdist(X1 / self.ls, X2 / self.ls, metric='euclidean')
        return self.var_f * (1 + np.sqrt(5)*r + 5/3*r**2) * np.exp(-np.sqrt(5)*r)

    def _fit(self):
        """Re-estimate kernel hyperparameters by marginal log-likelihood optimisation."""
        def neg_mll(log_params):
            ls, var_f = np.exp(log_params)
            self.ls, self.var_f = ls, var_f
            K = self._matern52(self.X_obs, self.X_obs) + self.noise * np.eye(len(self.X_obs))
            try:
                L  = np.linalg.cholesky(K)
                a  = np.linalg.solve(L.T, np.linalg.solve(L, self.y_obs))
                mll = -0.5 * self.y_obs @ a - np.sum(np.log(np.diag(L)))
                return -mll
            except np.linalg.LinAlgError:
                return 1e9
        res = minimize_scalar(lambda x: neg_mll([x, 0.0]),
                              bounds=(np.log(0.05), np.log(self.bounds[1]-self.bounds[0])),
                              method='bounded')
        self.ls = np.exp(res.x)

    def _predict(self, X_test):
        K    = self._matern52(self.X_obs, self.X_obs) + (self.noise + 1e-8) * np.eye(len(self.X_obs))
        K_s  = self._matern52(self.X_obs, X_test)
        K_ss = self._matern52(X_test, X_test)
        L    = np.linalg.cholesky(K)
        a    = np.linalg.solve(L.T, np.linalg.solve(L, self.y_obs))
        mu   = K_s.T @ a
        v    = np.linalg.solve(L, K_s)
        var  = np.diag(K_ss) - np.sum(v**2, axis=0)
        return mu, np.sqrt(np.maximum(var, 0))

    def _ei(self, X_test, xi: float = 0.01):
        mu, sigma = self._predict(X_test)
        best = np.max(self.y_obs)
        Z = (mu - best - xi) / (sigma + 1e-9)
        return (mu - best - xi) * stats.norm.cdf(Z) + sigma * stats.norm.pdf(Z)

    def suggest(self) -> float:
        if len(self.X_obs) < self.n_init:
            # Space-filling initialisation via Latin hypercube-style sampling
            i = len(self.X_obs)
            return self.bounds[0] + (i / (self.n_init - 1)) * (self.bounds[1] - self.bounds[0])
        self._fit()
        X_cand = np.linspace(*self.bounds, 200).reshape(-1, 1)
        ei = self._ei(X_cand)
        return float(X_cand[np.argmax(ei)])

    def update(self, x: float, y: float):
        self.X_obs = np.vstack([self.X_obs, [[x]]])
        self.y_obs = np.append(self.y_obs, y)

print('GaussianProcessBO defined.')

In [ ]:
# ── Define the LDS objective ───────────────────────────────────────────────────
# For each β we re-score the already-completed simulation metrics,
# so this is essentially free (no new simulations needed).

def recalculate_fitness_with_beta(
    row: pd.Series, beta: float, alpha: float = 0.5
) -> float:
    """
    Re-apply the Simulation._calculate_results() formula to stored metrics
    with a new beta, without re-running the simulation.
    
    Requires: sum_completed_time, sum_incomplete_elapsed_time,
              sum_incomplete_remaining_time, std_commute_time.
    """
    sum_completed     = row.get('sum_completed_time', 0.0)
    elapsed           = row.get('sum_incomplete_elapsed_time', 0.0)
    remaining         = row.get('sum_incomplete_remaining_time', 0.0)
    std_commute       = row.get('std_commute_time', 0.0)
    sum_incomplete    = elapsed + beta * remaining
    equity            = alpha * std_commute
    return sum_completed + sum_incomplete + equity

def landscape_discriminability_score(beta: float, df_results: pd.DataFrame, alpha: float = 0.5) -> float:
    """
    LDS(β) = σ²(fitness) / mean(β·sum_remaining)
    
    Numerator: how spread out the fitness landscape is (more = easier to optimise).
    Denominator: how much of that spread is just due to β scaling up penalties
                 (normalises away trivial variance inflation).
    """
    if 'sum_completed_time' not in df_results.columns:
        # Fallback if detailed breakdown wasn't collected: use stored fitness
        # and apply beta as a scaling adjustment on the penalty term.
        print("[WARN] Detailed time breakdown not found — using stored fitness as proxy.")
        return float(df_results['fitness_score'].var())

    scores = df_results.apply(lambda r: recalculate_fitness_with_beta(r, beta, alpha), axis=1).values
    penalty_contribution = beta * df_results['sum_incomplete_remaining_time'].values
    denominator = np.mean(penalty_contribution) if np.mean(penalty_contribution) > 0 else 1.0
    return float(np.var(scores) / denominator)

# Quick sanity check at the default β=2.0
lds_default = landscape_discriminability_score(2.0, df)
print(f'LDS at β=2.0 (default): {lds_default:.4f}')

In [ ]:
# ── Run the BO loop ───────────────────────────────────────────────────────────
BETA_BOUNDS  = (1.0, 15.0)
N_BO_ITERS   = 20         # typically converges in <15 iterations
ALPHA_FIXED  = BASE_CONFIG.get('ALPHA_STD_PENALTY', 0.5)

bo = GaussianProcessBO(bounds=BETA_BOUNDS, n_init=6)
bo_history = []

for iteration in range(N_BO_ITERS):
    beta_candidate = bo.suggest()
    lds = landscape_discriminability_score(beta_candidate, df, alpha=ALPHA_FIXED)
    bo.update(beta_candidate, lds)
    bo_history.append({'iteration': iteration, 'beta': beta_candidate, 'lds': lds})
    print(f'  iter {iteration+1:02d}  β={beta_candidate:.3f}  LDS={lds:.4f}')

bo_df  = pd.DataFrame(bo_history)
best_beta = float(bo.X_obs[np.argmax(bo.y_obs)])
best_lds  = float(np.max(bo.y_obs))
print(f'\n★  Best β found: {best_beta:.4f}  (LDS={best_lds:.4f})')

In [ ]:
# ── Visualise the BO results ──────────────────────────────────────────────────
beta_grid = np.linspace(*BETA_BOUNDS, 200)
lds_grid  = [landscape_discriminability_score(b, df, ALPHA_FIXED) for b in beta_grid]

X_test = beta_grid.reshape(-1, 1)
mu, sigma = bo._predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: GP posterior over LDS
ax = axes[0]
ax.plot(beta_grid, lds_grid, 'k-', lw=1.5, label='True LDS (computed)')
ax.plot(beta_grid, mu, 'steelblue', lw=1.5, label='GP posterior mean')
ax.fill_between(beta_grid, mu - 2*sigma, mu + 2*sigma, alpha=0.2, color='steelblue', label='GP ±2σ')
ax.scatter(bo_df['beta'], bo_df['lds'], c='darkorange', zorder=5, s=60, label='BO observations')
ax.axvline(best_beta, color='red', linestyle='--', lw=1.5, label=f'Best β={best_beta:.2f}')
ax.axvline(2.0, color='grey', linestyle=':', lw=1.2, label='Current β=2.0')
ax.set_xlabel('β  (underserved passenger penalty)')
ax.set_ylabel('Landscape Discriminability Score (LDS)')
ax.set_title('Bayesian Optimisation over β', fontweight='bold')
ax.legend(fontsize=8)

# Right: convergence plot
ax = axes[1]
running_best = bo_df['lds'].cummax()
ax.plot(bo_df['iteration'] + 1, bo_df['lds'], 'o--', color='darkorange', alpha=0.6, label='LDS per iteration')
ax.plot(bo_df['iteration'] + 1, running_best, 'r-', lw=2, label='Running best')
ax.axhline(lds_default, color='grey', linestyle=':', lw=1.2, label=f'LDS at β=2.0 (default)')
ax.set_xlabel('BO Iteration')
ax.set_ylabel('LDS')
ax.set_title('BO Convergence', fontweight='bold')
ax.legend(fontsize=9)

plt.suptitle('β Penalty Calibration via Bayesian Optimisation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig4_beta_bo.png', bbox_inches='tight')
plt.show()

---
## 6. Effect of β on Fitness Landscape Shape

We visualise how the distribution of fitness scores across configurations changes as β varies, to confirm the BO finding is interpretable and not a statistical artefact.

In [ ]:
beta_probe_values = [1.0, 2.0, best_beta, 5.0, 10.0]
fitness_distributions = {}

for b in beta_probe_values:
    if 'sum_completed_time' in df.columns:
        scores = df.apply(lambda r: recalculate_fitness_with_beta(r, b, ALPHA_FIXED), axis=1).values
    else:
        scores = df['fitness_score'].values  # fallback
    fitness_distributions[b] = scores

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: KDE of fitness distributions
ax = axes[0]
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(beta_probe_values)))
for (b, scores), c in zip(fitness_distributions.items(), colors):
    label = f'β={b:.1f}' + (' ★ optimal' if abs(b - best_beta) < 0.05 else '')
    label += ' (default)' if abs(b - 2.0) < 0.05 else ''
    scores_norm = MinMaxScaler().fit_transform(scores.reshape(-1,1)).flatten()
    sns.kdeplot(scores_norm, ax=ax, label=label, color=c, lw=2)
ax.set_xlabel('Normalised Fitness Score')
ax.set_ylabel('Density')
ax.set_title('Fitness Distribution Shape by β\n(wider = more discriminating)', fontweight='bold')
ax.legend(fontsize=9)

# Right: variance and IQR vs β
beta_sweep = np.linspace(*BETA_BOUNDS, 60)
variances, iqrs = [], []
for b in beta_sweep:
    if 'sum_completed_time' in df.columns:
        s = df.apply(lambda r: recalculate_fitness_with_beta(r, b, ALPHA_FIXED), axis=1).values
    else:
        s = df['fitness_score'].values
    variances.append(np.var(s))
    iqrs.append(np.percentile(s, 75) - np.percentile(s, 25))

ax2 = axes[1]
color1 = 'steelblue'
ax2.plot(beta_sweep, variances, color=color1, lw=2, label='Variance')
ax2.set_xlabel('β')
ax2.set_ylabel('Fitness Variance', color=color1)
ax2.tick_params(axis='y', labelcolor=color1)
ax2r = ax2.twinx()
ax2r.plot(beta_sweep, iqrs, color='darkorange', lw=2, linestyle='--', label='IQR')
ax2r.set_ylabel('IQR', color='darkorange')
ax2r.tick_params(axis='y', labelcolor='darkorange')
ax2.axvline(best_beta, color='red', lw=1.5, linestyle='--', label=f'Optimal β={best_beta:.2f}')
ax2.axvline(2.0, color='grey', lw=1.2, linestyle=':', label='Default β=2.0')
ax2.set_title('Fitness Spread vs β', fontweight='bold')
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2r.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, fontsize=9)

plt.suptitle('Fitness Landscape Shape as a Function of β', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig5_beta_landscape.png', bbox_inches='tight')
plt.show()

---
## 7. Proposed Revised `StaticSurrogateEvaluator`

Based on the analysis above, we present a drop-in replacement for the `evaluate()` method that incorporates:
- Coverage normalisation (λ parameter)
- Explicit demand-weighted unreachable-OD penalty (γ parameter)
- Optional equity term mirroring α·std(path_costs) from the microscopic evaluator

All parameters default to the optimised values found above.

In [ ]:
import inspect
import textwrap

revised_evaluator_code = textwrap.dedent(f'''
class StaticSurrogateEvaluatorV2(StaticSurrogateEvaluator):
    """
    Revised static surrogate with improved rank-agreement against microscopic fitness.

    Changes from V1
    ---------------
    1. Coverage normalisation (λ): routing cost is divided by completed^λ so that
       route systems that leave many OD pairs unreachable are penalised multiplicatively,
       not just additively.
    2. Explicit unreachable-OD penalty (γ): scaled to mean routed path weight rather
       than the hard-coded 100,000, so the penalty remains proportional to the
       cost distribution of the route system.
    3. Equity term (α_surr): std(path_weights) of successfully routed pairs mirrors
       the α·std_commute term in the microscopic evaluator.

    Calibrated defaults (from differential evolution on {N_CONFIGS} paired configs):
      λ = {best_lambda:.4f}
      γ_scale = {best_gamma:.2f}   (multiplied by mean_weight / n_samples)
      δ = {best_delta:.6f}
      α_surr = {ALPHA_FIXED:.4f}   (matches ALPHA_STD_PENALTY in base config)
    """

    def __init__(
        self,
        config: dict,
        city_graph,
        demand_sampler,
        num_samples: int = 500,
        lambda_coverage: float = {best_lambda:.4f},
        gamma_unreachable: float = {best_gamma:.2f},
        delta_fleet: float = {best_delta:.6f},
        alpha_equity: float = {ALPHA_FIXED:.4f},
    ) -> None:
        super().__init__(config, city_graph, demand_sampler, num_samples)
        self.lambda_coverage   = lambda_coverage
        self.gamma_unreachable = gamma_unreachable
        self.delta_fleet       = delta_fleet
        self.alpha_equity      = alpha_equity

    def evaluate(self, routes, verbose: bool = False):
        import uuid, math
        from utils.simulation import SimulationResult
        from utils.travel_graph import TravelGraph

        tg_cfg = self.config.get("travel_graph", {{}}).copy()
        base_transfer = tg_cfg.get("transfer_wt", 5.0)
        tg_cfg["transfer_wt"] = base_transfer * 2.5

        tg = TravelGraph(cg=self.city_graph, config=tg_cfg, routes=routes)

        weights, completed = [], 0
        n_unreachable = 0

        for start, end in self.od_pairs:
            path = tg.findShortestJourney(start, end)
            if path:
                w = sum(e.weight for e in path)
                if 0 < w < float("inf"):
                    weights.append(w)
                    completed += 1
                    continue
            n_unreachable += 1

        # --- Revised composite score ---
        n_samples    = len(self.od_pairs)
        total_weight = sum(weights)
        mean_weight  = total_weight / max(completed, 1)

        # 1. Coverage-normalised routing cost
        routing_term  = total_weight / max(completed, 1) ** self.lambda_coverage

        # 2. Unreachable OD penalty proportional to mean path weight
        unreachable_unit_cost = mean_weight  # scales with route quality
        coverage_term = self.gamma_unreachable * n_unreachable * unreachable_unit_cost / max(n_samples, 1)

        # 3. Fleet operational cost
        fleet_cost    = sum(sum(e.getLength() for e in r.path) for r in routes)
        fleet_term    = self.delta_fleet * fleet_cost

        # 4. Equity (variance of path costs among routed ODs)
        if len(weights) > 1:
            std_w    = math.sqrt(sum((w - mean_weight)**2 for w in weights) / len(weights))
            equity_term = self.alpha_equity * std_w
        else:
            equity_term = 0.0

        surrogate_cost = routing_term + coverage_term + fleet_term + equity_term

        return SimulationResult(
            surrogate_cost=surrogate_cost,
            metrics={{
                "passenger_routing_cost" : total_weight,
                "routing_term"           : routing_term,
                "coverage_term"          : coverage_term,
                "fleet_operational_cost" : fleet_cost,
                "equity_term"            : equity_term,
                "surrogate_samples"      : n_samples,
                "completed_routes"       : completed,
                "unreachable_count"      : n_unreachable,
                "coverage_ratio"         : completed / max(n_samples, 1),
            }},
            recorded_paths=[],
            jeep_system=None,
            sim_id=f"SURRV2{{uuid.uuid4().hex[:6]}}",
            score_kind="surrogate"
        )
''')

print(revised_evaluator_code)

---
## 8. Validate the Revised Surrogate

Run the revised evaluator on the same route configurations and compare rank agreement.

In [ ]:
# Define and instantiate the revised class
exec(revised_evaluator_code)

surrogate_v2 = StaticSurrogateEvaluatorV2(
    config=BASE_CONFIG,
    city_graph=cg,
    demand_sampler=ddm,
    num_samples=SURROGATE_OD,
    lambda_coverage=best_lambda,
    gamma_unreachable=best_gamma,
    delta_fleet=best_delta,
    alpha_equity=ALPHA_FIXED
)

# Re-evaluate with V2 (re-uses the same route_systems list from Section 1B)
v2_scores = []
for routes in tqdm(route_systems, desc='V2 surrogate evaluation'):
    r = surrogate_v2.evaluate(routes)
    v2_scores.append(r.surrogate_cost)

df['surrogate_v2_cost'] = v2_scores

v2_baseline = correlation_report(df['surrogate_cost'].values, df['fitness_score'].values)
v2_revised  = correlation_report(df['surrogate_v2_cost'].values, df['fitness_score'].values)

summary = pd.DataFrame([
    {'Evaluator': 'V1 (current)',  **{k: v for k, v in v2_baseline.items() if k in ['pearson_r','spearman_rho','kendall_tau','rank_rmse']}},
    {'Evaluator': 'V2 (revised)', **{k: v for k, v in v2_revised.items()  if k in ['pearson_r','spearman_rho','kendall_tau','rank_rmse']}},
])
print(summary.to_string(index=False, float_format='{:.4f}'.format))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
rank_fit = stats.rankdata(df['fitness_score'])

for ax, col, label, color in [
    (axes[0], 'surrogate_cost',    'V1 (Current)',  'steelblue'),
    (axes[1], 'surrogate_v2_cost', 'V2 (Revised)', 'seagreen'),
]:
    rank_s = stats.rankdata(df[col])
    rho, _ = spearmanr(df[col], df['fitness_score'])
    ax.scatter(rank_s, rank_fit, s=50, alpha=0.75, edgecolors='white', lw=0.4, color=color)
    ax.plot([1, len(df)], [1, len(df)], 'k--', lw=1.2, alpha=0.5)
    ax.set_title(f'{label}\nSpearman ρ={rho:.3f}', fontweight='bold')
    ax.set_xlabel('Surrogate Rank')
    ax.set_ylabel('Fitness Rank')

plt.suptitle('Rank Agreement: V1 vs V2 Surrogate', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fig6_v1_vs_v2.png', bbox_inches='tight')
plt.show()

---
## 9. Summary of Findings

In [ ]:
print('=' * 65)
print('  CALIBRATION STUDY — SUMMARY')
print('=' * 65)

print(f'''
§ 1. Surrogate Validity (Baseline)
   Pearson r     = {v2_baseline["pearson_r"]:.4f}  (explains {v2_baseline["pearson_r"]**2*100:.1f}% of fitness variance)
   Spearman ρ    = {v2_baseline["spearman_rho"]:.4f}
   Kendall τ     = {v2_baseline["kendall_tau"]:.4f}
   Rank RMSE     = {v2_baseline["rank_rmse"]:.2f} positions

   Root cause: V1 surrogate ignores coverage (unreachable ODs receive
   a flat penalty rather than a proportional one), omits the equity
   term entirely, and does not normalise routing cost by coverage.

§ 2. Improved Surrogate (V2)
   Pearson r     = {v2_revised["pearson_r"]:.4f}
   Spearman ρ    = {v2_revised["spearman_rho"]:.4f}  (+{(v2_revised["spearman_rho"]-v2_baseline["spearman_rho"])*100:.1f} pp improvement)
   Kendall τ     = {v2_revised["kendall_tau"]:.4f}
   Rank RMSE     = {v2_revised["rank_rmse"]:.2f} positions

   Parameters: λ={best_lambda:.3f}, γ={best_gamma:.1f}, δ={best_delta:.5f},
               α_equity={ALPHA_FIXED:.3f}

§ 3. β Penalty Calibration (Bayesian Optimisation)
   Current β     = 2.00  (LDS = {lds_default:.4f})
   Optimal β     = {best_beta:.4f}  (LDS = {best_lds:.4f})
   Improvement   = {(best_lds - lds_default) / lds_default * 100:+.1f}% more discriminating landscape

§ 4. Recommended Config Changes
   BETA_PENALTY        : 2.0  →  {best_beta:.2f}
   ALPHA_STD_PENALTY   : unchanged ({ALPHA_FIXED:.2f})
   StaticSurrogateEvaluator: replace with V2 (see Section 7 code)
''')
print('=' * 65)

In [ ]:
# Export revised evaluator code to a standalone .py file
output_path = RESULTS_DIR / 'static_surrogate_v2.py'
header = '# Auto-generated by surrogate_calibration_study.ipynb\n'\
         '# Drop this class into utils/simulation.py and swap StaticSurrogateEvaluator\n\n'\
         'import math, uuid\n'\
         'from utils.simulation import SimulationResult, StaticSurrogateEvaluator\n'\
         'from utils.travel_graph import TravelGraph\n\n'
with open(output_path, 'w') as f:
    f.write(header + revised_evaluator_code)
print(f'Saved to {output_path}')